In [0]:
from pyspark.sql.functions import (
    col, lit, sequence, explode,
    date_format, dayofmonth, month, year,
    dayofweek, weekofyear, quarter,
    dayofyear, last_day, trunc,
    current_date, current_timestamp,
    when, add_months, concat, lpad, 
    weekofyear, year
)

# --------------------------------------------------
# Configuration
# --------------------------------------------------
START_DATE = "2011-01-01"
TARGET_TABLE = "silver_dev.global_mart_retail.dim_date"

# --------------------------------------------------
# Generate date spine (2011-01-01 → today)
# --------------------------------------------------
date_df = (
    spark
    .createDataFrame([(START_DATE,)], ["start_date"])
    .select(
        explode(
            sequence(
                lit(START_DATE).cast("date"),
                current_date(),
                lit(1).cast("interval day")
            )
        ).alias("date")
    )
)

# --------------------------------------------------
# Build Kimball-compliant Dim_Date
# --------------------------------------------------
dim_date_df = (
    date_df
    .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast("int"))

    .withColumn("day", dayofmonth(col("date")))
    .withColumn("day_name", date_format(col("date"), "EEEE"))
    .withColumn("day_of_week", dayofweek(col("date")))
    .withColumn("day_of_year", dayofyear(col("date")))

    .withColumn("week_of_year", weekofyear(col("date")))
    .withColumn(
            "year_week",
            concat(
                year(col("date")),
                lit("-"),
                lpad(weekofyear(col("date")), 2, "0")
            )
        )

    .withColumn("month", month(col("date")))
    .withColumn("month_name", date_format(col("date"), "MMMM"))
    .withColumn("year_month", date_format(col("date"), "yyyy-MM"))

    .withColumn("quarter", quarter(col("date")))
    .withColumn("quarter_name", date_format(col("date"), "'Q'q"))

    .withColumn("year", year(col("date")))

    .withColumn("is_weekend", when(col("day_of_week").isin([1, 7]), True).otherwise(False))
    .withColumn("is_weekday", when(col("day_of_week").isin([2, 3, 4, 5, 6]), True).otherwise(False))

    .withColumn("is_month_start", col("date") == trunc(col("date"), "MM"))
    .withColumn("is_month_end", col("date") == last_day(col("date")))

    .withColumn("is_quarter_start", col("date") == trunc(col("date"), "Q"))
    .withColumn(
        "is_quarter_end",
        col("date") == last_day(add_months(trunc(col("date"), "Q"), 2))
    )

    .withColumn("is_year_start", col("date") == trunc(col("date"), "YYYY"))
    .withColumn(
        "is_year_end",
        col("date") == last_day(add_months(trunc(col("date"), "YYYY"), 11))
    )

    .withColumn("load_timestamp", current_timestamp())
)

# --------------------------------------------------
# Write to Silver (static dimension → overwrite)
# --------------------------------------------------
(
    dim_date_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)


In [0]:
%sql
select current_date, * from silver_dev.global_mart_retail.dim_date where date >= current_date